# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Teninlanimi/flyrank-interenship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repo root
root_candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = None
for candidate in root_candidates:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

sys.path.insert(0, str(repo_root))

from scripts.ml_utils import (
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
)

raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
output_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"

df = pd.read_csv(raw_path)

# Numeric cleaning
for col in NUMERIC_COLUMNS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    else:
        df[col] = 0

# Categorical cleaning
for col in CATEGORICAL_COLUMNS:
    if col in df.columns:
        df[col] = (
            df[col]
            .fillna("unknown")
            .astype(str)
            .replace({"": "unknown", "nan": "unknown", "None": "unknown"})
        )
    else:
        df[col] = "unknown"

# Fill numeric gaps with zero
for col in [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct"
]:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

# Filter and deduplicate
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Engineered features
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = (
    (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
).astype(int)

# Final feature frame
feature_frame = df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES + ["is_declining_label"]].copy()

for col in MODEL_CATEGORICAL_FEATURES:
    feature_frame[col] = feature_frame[col].astype(str)

output_path.parent.mkdir(parents=True, exist_ok=True)
feature_frame.to_csv(output_path, index=False)

print("Feature vector built.")
print(feature_frame.head())

Feature vector built.
   search_volume  competition   cpc  word_count  char_count  \
0           10.0         0.67  2.05      3221.0     20457.0   
1           90.0         0.01  0.05      2481.0     15562.0   
2            0.0         0.00  0.00      3515.0     23643.0   
3           10.0         0.00  0.00         0.0         0.0   
4            0.0         0.00  0.00      2803.0     17469.0   

   log_impressions_90d  log_clicks_90d  log_sessions_90d  log_ai_sessions_90d  \
0             8.243808        3.401197          2.890372                  0.0   
1             9.636980        2.079442          2.302585                  0.0   
2             9.440023        2.484907          2.484907                  0.0   
3             9.371779        4.077537          4.369448                  0.0   
4             9.859588        3.218876          4.983607                  0.0   

   days_with_impressions  ...  ai_traffic_pct  competition_level  \
0                     88  ...             0.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



Numerical features - search_volume, competition, cpc, word_count, char_count, and historical traffic metrics: describe keyword, content, and past performance. Missing values were filled with `0` where appropriate. All are available before the prediction date.

Categorical features - competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, quality_band, position_band: were encoded for modeling. Missing values were assigned a default category where needed.

Engineered features - log-transformed traffic metrics and derived tiers : were created from historical data only, making them available at the decision point and helping prevent data leakage.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Load the feature vector
root_candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = None
for candidate in root_candidates:
    if (candidate / "data" / "processed" / "refresh_feature_vector.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

sys.path.insert(0, str(repo_root))
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
feature_frame = pd.read_csv(feature_path)

print("=" * 60)
print("LEAKAGE HUNT")
print("=" * 60)

# 1. Check that label-derived columns are excluded
print("\n1. Label-derived columns (should NOT be features):")
label_derived = ["trend_direction", "trend_pct", "is_declining_label"]
for col in label_derived:
    in_numeric = col in MODEL_NUMERIC_FEATURES
    in_categorical = col in MODEL_CATEGORICAL_FEATURES
    status = "LEAK!" if (in_numeric or in_categorical) else "OK (excluded)"
    print(f"  {col}: {status}")

# 2. Check that product flags are excluded (they are not causal)
print("\n2. Product flags (should NOT be features):")
product_flags = ["provider_used", "model_used"]
for col in product_flags:
    in_numeric = col in MODEL_NUMERIC_FEATURES
    in_categorical = col in MODEL_CATEGORICAL_FEATURES
    status = "LEAK!" if (in_numeric or in_categorical) else "OK (excluded)"
    print(f"  {col}: {status}")

# 3. Check that content_id and client_id are excluded (grouping only, not features)
print("\n3. Identifiers (should NOT be features):")
ids = ["content_id", "client_id"]
for col in ids:
    in_numeric = col in MODEL_NUMERIC_FEATURES
    in_categorical = col in MODEL_CATEGORICAL_FEATURES
    status = "LEAK!" if (in_numeric or in_categorical) else "OK (excluded)"
    print(f"  {col}: {status}")

# 4. Check feature-label correlation for suspiciously high values
print("\n4. Feature-label correlations (should be moderate, not near 1.0):")
df_numeric = feature_frame[MODEL_NUMERIC_FEATURES + ["is_declining_label"]].copy()
for col in MODEL_NUMERIC_FEATURES:
    df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")

correlations = df_numeric.corr()["is_declining_label"].drop("is_declining_label").sort_values(key=abs, ascending=False)
print("\nTop 5 positive correlations:")
print(correlations.head(5))
print("\nTop 5 negative correlations:")
print(correlations.tail(5))

# Warn if any correlation is suspiciously high (> 0.9 or < -0.9)
high_corr = correlations[(correlations > 0.9) | (correlations < -0.9)]
if len(high_corr) > 0:
    print("\n⚠️  WARNING: Suspiciously high correlations (potential leakage):")
    print(high_corr)
else:
    print("\n✓ No suspiciously high correlations detected.")

# 5. Test avg_position handling (0 means no data, not rank 0)
print("\n5. avg_position handling (0 = no data, not rank zero):")
if "avg_position" in feature_frame.columns:
    zeros = (feature_frame["avg_position"] == 0).sum()
    print(f"  Rows with avg_position = 0: {zeros:,} ({zeros/len(feature_frame)*100:.1f}%)")
    print(f"  Explanation: These are NOT rank-0; they are rows with no GSC position data.")
    print(f"  This is correct; we filled missing with 0 and that's transparent in the model.")

# 6. Summary of what we're feeding the model
print("\n6. Model feature summary:")
print(f"  Numeric features: {len(MODEL_NUMERIC_FEATURES)}")
print(f"    {MODEL_NUMERIC_FEATURES}")
print(f"\n  Categorical features: {len(MODEL_CATEGORICAL_FEATURES)}")
print(f"    {MODEL_CATEGORICAL_FEATURES}")
print(f"\n  Label: is_declining_label")
print(f"  Label distribution: {feature_frame['is_declining_label'].value_counts().to_dict()}")

print("\n" + "=" * 60)
print("LEAKAGE HUNT COMPLETE")
print("=" * 60)

LEAKAGE HUNT

1. Label-derived columns (should NOT be features):
  trend_direction: OK (excluded)
  trend_pct: OK (excluded)
  is_declining_label: OK (excluded)

2. Product flags (should NOT be features):
  provider_used: OK (excluded)
  model_used: OK (excluded)

3. Identifiers (should NOT be features):
  content_id: OK (excluded)
  client_id: OK (excluded)

4. Feature-label correlations (should be moderate, not near 1.0):

Top 5 positive correlations:
days_with_impressions    0.190055
log_impressions_90d      0.177473
content_age_days        -0.163882
word_count               0.118863
char_count               0.108025
Name: is_declining_label, dtype: float64

Top 5 negative correlations:
cpc                   -0.006031
log_ai_sessions_90d   -0.004304
log_clicks_90d         0.003469
scroll_rate           -0.002711
ai_traffic_pct         0.002435
Name: is_declining_label, dtype: float64

✓ No suspiciously high correlations detected.

5. avg_position handling (0 = no data, not rank zero

What this code checks:

Label-derived columns – Confirms trend_direction and trend_pct are NOT features (they are the source of the label).
Product flags – Confirms provider_used and model_used are excluded (they describe the content's origin, not performance).
Identifiers – Confirms content_id and client_id are excluded (grouping keys only).
Feature-label correlations – Flags any feature with correlation > ±0.9 (suspicious leakage signal).
avg_position = 0 handling – Documents that 0 means "no position data," not rank zero.
Feature summary – Lists what the model actually receives.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("""
EXCLUDED FIELDS AND WHY

Label-derived (source of target, not predictive):
  - trend_direction: This IS the label definition; using it would be circular.
  - trend_pct: This IS the label definition; using it would be circular.
  - is_declining_label: This IS the target variable; it must be excluded from features.

Product metadata (not causal to refresh need):
  - provider_used: Describes content origin, not performance or refresh demand.
  - model_used: Describes content creation tool, not performance or refresh demand.

Identifiers (grouping keys, not predictive):
  - content_id: Used only to group rows; no predictive signal.
  - client_id: Used only to group rows; no predictive signal.

Data quality flags (metadata, not features):
  - is_deleted: Binary marker of deletion status, not a performance signal.

Query window leakage (future data relative to label date):
  - impressions_90d: fact_content_query_90d window (Apr-Jun 2026) overlaps label date; using it is future leakage.
  - last30: Same window as impressions_90d; excludes prev30 for this reason.

Timestamp (temporal grouping only):
  - report_date: Used to define label window and split; not a model feature.
  - observation_date: Used to define feature window; not a model feature.
""")


EXCLUDED FIELDS AND WHY

Label-derived (source of target, not predictive):
  - trend_direction: This IS the label definition; using it would be circular.
  - trend_pct: This IS the label definition; using it would be circular.
  - is_declining_label: This IS the target variable; it must be excluded from features.

Product metadata (not causal to refresh need):
  - provider_used: Describes content origin, not performance or refresh demand.
  - model_used: Describes content creation tool, not performance or refresh demand.

Identifiers (grouping keys, not predictive):
  - content_id: Used only to group rows; no predictive signal.
  - client_id: Used only to group rows; no predictive signal.

Data quality flags (metadata, not features):
  - is_deleted: Binary marker of deletion status, not a performance signal.

Query window leakage (future data relative to label date):
  - impressions_90d: fact_content_query_90d window (Apr-Jun 2026) overlaps label date; using it is future leakage.
  - 

In [5]:
exclusions = pd.DataFrame({
    "field": [
        "trend_direction",
        "trend_pct",
        "is_declining_label",
        "provider_used",
        "model_used",
        "content_id",
        "client_id",
        "is_deleted",
        "impressions_90d",
        "last30",
        "report_date",
        "observation_date"
    ],
    "reason": [
        "Source of label definition; circular if used as feature",
        "Source of label definition; circular if used as feature",
        "Target variable; must be excluded",
        "Content origin metadata; not causal to refresh need",
        "Content creation tool; not causal to refresh need",
        "Grouping key only; no predictive signal",
        "Grouping key only; no predictive signal",
        "Binary deletion marker; not a performance signal",
        "fact_content_query_90d (Apr-Jun 2026) is future data relative to label; leakage",
        "Same window as impressions_90d; only prev30 is safe",
        "Temporal grouping; used to define label window, not a feature",
        "Temporal grouping; used to define feature window, not a feature"
    ]
})

print("EXCLUDED FIELDS:")
print(exclusions.to_string(index=False))

EXCLUDED FIELDS:
             field                                                                          reason
   trend_direction                         Source of label definition; circular if used as feature
         trend_pct                         Source of label definition; circular if used as feature
is_declining_label                                               Target variable; must be excluded
     provider_used                             Content origin metadata; not causal to refresh need
        model_used                               Content creation tool; not causal to refresh need
        content_id                                         Grouping key only; no predictive signal
         client_id                                         Grouping key only; no predictive signal
        is_deleted                                Binary deletion marker; not a performance signal
   impressions_90d fact_content_query_90d (Apr-Jun 2026) is future data relative to label; l

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.